**Tensor 基础与 GPU 操作**：所有深度学习任务（CNN、RNN、BERT）的数据预处理都依赖这套语法，是后续 Dataset、DataLoader、模型训练的前置基础。

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ── 创建 Tensor ──────────────────────────────────────
x = torch.tensor([1.0, 2.0, 3.0])
x = torch.zeros(3, 4)
x = torch.ones(2, 3, 4)
x = torch.randn(10, 5)           # 标准正态
x = torch.arange(12).reshape(3, 4)
x = torch.eye(4)                  # 单位矩阵

# ── 数据类型 ──────────────────────────────────────────
x = torch.tensor([1, 2, 3], dtype=torch.float32)   # float32（最常用）
x = torch.tensor([1, 2, 3], dtype=torch.long)       # int64（标签）
x = x.float()      # 转 float32
x = x.long()       # 转 int64（标签必须是 long）

# ── GPU 操作 ──────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
x = x.to(device)               # 将 tensor 移到 GPU
# model = model.to(device)       # 将模型移到 GPU

# ── 与 NumPy 互转 ──────────────────────────────────────
np_arr = np.random.randn(3, 4)
tensor = torch.from_numpy(np_arr)         # NumPy → Tensor（共享内存！）
np_back = tensor.cpu().detach().numpy()   # Tensor → NumPy（先到CPU，再detach）

使用设备: cpu


**自动微分**:
深度学习标准训练四步就是基于这套梯度机制：

1. 前向传播算损失；
2. `loss.backward()`自动求梯度；
3. `optimizer.step()`依靠梯度更新参数；
4. `optimizer.zero_grad()`清空梯度，准备下一轮迭代。

In [10]:
# ── 计算图与梯度 ──────────────────────────────────────
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# 构建计算图
z = x**2 + 2*x*y + y**3   # z = x^2 + 2xy + y^3

# 反向传播，计算梯度
z.backward()

print(x.grad)   # dz/dx = 2x + 2y = 4 + 6 = 10
print(y.grad)   # dz/dy = 2x + 3y^2 = 4 + 27 = 31

# ── 注意事项 ──────────────────────────────────────────
# 每次 backward 后梯度会累积，训练中必须 zero_grad()
x.grad.zero_()   # 或 optimizer.zero_grad()

# 不需要梯度的代码（推理阶段）用 torch.no_grad()
# with torch.no_grad():
    # output = model(input_tensor)   # 不构建计算图，节省内存

tensor(10.)
tensor(31.)


tensor(0.)

**nn.Module —— 用面向对象方式构建网络**：代码展示 PyTorch 搭建神经网络的两种主流写法：`nn.Sequential`序列容器、继承`nn.Module`自定义类；分别实现多层全连接 MLP、双向双层 LSTM 文本分类网络，附带可训练参数量统计，是深度学习搭建网络的标准两套范式，适配简单网络与复杂定制化网络。

In [12]:
import torch.nn as nn

# ── 方式1：使用 nn.Sequential（简单网络）──────────────
mlp = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

# ── 方式2：继承 nn.Module（复杂网络，推荐）──────────────
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))   # (batch, seq, embed_dim)
        output, (hidden, _) = self.lstm(embedded)    # output: (batch, seq, hidden*2)
        # 取最后一个时间步的前向和后向 hidden state 拼接
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, hidden*2)
        return self.classifier(self.dropout(hidden_cat))

# 查看模型结构和参数量
model = TextClassifier(30000, 128, 256, 2)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"可训练参数量: {total_params:,}")

TextClassifier(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=512, out_features=2, bias=True)
)
可训练参数量: 6,208,514


**完整训练流程模板**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ===================== 1. 基础配置 =====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
num_epochs = 20

# ===================== 2. 模型定义（可替换为你自己的模型） =====================
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=784, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = x.flatten(1)  # 图像展平为向量，适配全连接层
        return self.net(x)

model = SimpleClassifier().to(device)

# ===================== 3. 数据集与DataLoader（可替换为你自己的数据集） =====================
# 示例用MNIST手写数字数据集，自动下载
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ===================== 4. 损失函数 & 优化器（你之前缺失的核心部分） =====================
criterion = nn.CrossEntropyLoss()  # 分类任务标准交叉熵损失
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # Adam优化器

# ===================== 5. 训练/评估函数 =====================
def train_epoch(model, loader, optimizer, criterion, device):
    """训练一个 epoch，返回平均 loss 和 accuracy"""
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()              # 步骤1：清空梯度
        outputs = model(batch_x)           # 步骤2：前向传播
        loss = criterion(outputs, batch_y) # 步骤3：计算损失
        loss.backward()                    # 步骤4：反向传播
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()                   # 步骤5：更新参数

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / len(loader), correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    """评估，返回 loss, accuracy, 所有预测结果"""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        probs = torch.softmax(outputs, dim=1)

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels, all_probs

# ===================== 6. 训练主循环 =====================
best_val_acc = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

# 学习率调度器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _, _ = eval_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)   # 根据验证 loss 调整学习率

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # 保存最优模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  → 保存最优模型 (val_acc = {best_val_acc:.4f})")

# ===================== 7. 加载最优模型并测试 =====================
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
test_loss, test_acc, test_preds, test_labels, test_probs = \
    eval_epoch(model, test_loader, criterion, device)
print(f"\n最终测试结果：Loss={test_loss:.4f}, Accuracy={test_acc:.4f}")